# Milestone 0: look at the data

**Goal:** find out what the source text and audio actually are before writing
anything that depends on them. See `PLAN.md`, section 4, milestone 0.

By the end you will have:
- the World English Bible text and the audio for every chapter in the splits, saved to Google Drive;
- listened to the reader and written down every quirk;
- checked format, sample rate and where the audio's frequency content stops;
- the hours per split, so the train list can be adjusted to about 10 hours;
- heard a mel-spectrogram turned back into audio with Griffin-Lim.

**Runtime:** CPU is enough (Runtime → Change runtime type → CPU).

**Gate to milestone 1:** one reader throughout, clean enough audio, spoken text
matches written text, and the notes cell at the bottom filled in.

**Background reading:** Valerio Velardo, "Audio Signal Processing for ML"
(YouTube), the videos on sampling, the Fourier transform and mel-spectrograms.

## Setup

In [ ]:
# Get the code, install it, and connect Google Drive.
import subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/nicholas-on/TTS.git"
BRANCH = "main"
CODE_DIR = Path("/content/TTS")

def github_token():
    """The repository is private, so cloning needs a GitHub token, stored as a
    Colab secret named GITHUB_TOKEN with "Notebook access" switched on.
    See the README, "Running the proof of concept on Google Colab"."""
    try:
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except ImportError:
        return None  # not in Colab
    except Exception as e:  # secret missing, or notebook access not granted
        print(f"No GITHUB_TOKEN available ({type(e).__name__}).")
        return None

def git(*args):
    result = subprocess.run(["git", *args], capture_output=True, text=True)
    if result.returncode != 0:
        message = result.stderr.replace(token, "***") if token else result.stderr
        raise RuntimeError(
            f"git {args[0]} failed:\n{message}\n"
            "If this says 'Repository not found' or 'could not read Username', the token is "
            "missing or wrong: add a Colab secret named GITHUB_TOKEN (key icon, left sidebar), "
            "switch on 'Notebook access', and rerun this cell.")

token = github_token()
if not CODE_DIR.exists():
    url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL
    git("clone", "-q", "-b", BRANCH, url, str(CODE_DIR))
else:
    git("-C", str(CODE_DIR), "pull", "-q")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(CODE_DIR)], check=True)
if str(CODE_DIR / "src") not in sys.path:
    sys.path.insert(0, str(CODE_DIR / "src"))

from bibletts.colab import mount_drive, gpu_name, disk_free_gb
from bibletts.config import load_config

mount_drive()
cfg = load_config(CODE_DIR / "configs" / "data.yaml")
DRIVE = Path(cfg["paths"]["drive_root"])   # durable
LOCAL = Path(cfg["paths"]["local_root"])   # fast, wiped when the session ends
RAW = DRIVE / "raw"                        # downloaded originals
for d in (DRIVE, LOCAL, RAW):
    d.mkdir(parents=True, exist_ok=True)
print("GPU:", gpu_name(), "| free local disk:", round(disk_free_gb()), "GB")

## 1. The text

eBible publishes the World English Bible in several formats. We use
verse-per-line: one verse per line, starting with a book code, chapter and
verse, such as `MAT 5:3 Blessed are the poor in spirit...`.

If the download fails or the format looks different, open
<https://ebible.org/find/details.php?id=eng-web> in a browser, find the
verse-per-line (VPL) download, and put its URL in `configs/data.yaml` under
`source.text_url`.

In [ ]:
from bibletts.download import download, extract_zip
from bibletts.corpus import parse_vpl

text_url = cfg["source"]["text_url"]
zip_path = download(text_url, RAW / "text" / Path(text_url).name)
files = extract_zip(zip_path, RAW / "text")
print("Files in the zip:", *[f.name for f in files], sep="\n  ")

txts = [f for f in files if f.suffix.lower() == ".txt"]
vpl_path = max(txts, key=lambda f: ("vpl" in f.name.lower(), f.stat().st_size))
print("\nUsing", vpl_path.name, "- first lines:")
with open(vpl_path, encoding="utf-8-sig") as f:
    for _ in range(5):
        print("  ", f.readline().rstrip()[:120])

In [ ]:
chapters_text = parse_vpl(vpl_path)
books_found = sorted({b for b, _ in chapters_text})
n_verses = sum(len(v) for v in chapters_text.values())
print(f"{len(chapters_text)} chapters, {n_verses} verses, {len(books_found)} books")
assert len(chapters_text) > 1000, "Parsed far too little: the file format is not what parse_vpl expects."

print("\nMatthew 5:1-5:")
for verse, raw in chapters_text[("MAT", 5)][:5]:
    print(f"  {verse:>2}  {raw}")

### What the cleaner will do to the text

The models see cleaned text: lowercase letters, apostrophes, single spaces, and
the punctuation marks `, ; . ? !`. Look at the unusual characters below. Anything
that would be read aloud (for example `&` read as "and") needs a rule in
`src/bibletts/text.py`.

In [ ]:
from collections import Counter
from bibletts.text import clean_text

alphabet, strip = cfg["text"]["alphabet"], cfg["text"]["strip_accents"]
dropped = Counter()
digits = []
for key, verses in chapters_text.items():
    for v, raw in verses:
        clean_text(raw, alphabet, strip, dropped)
        if any(c.isdigit() for c in raw):
            digits.append((key, v, raw))

print("Characters the cleaner drops (whole Bible):", dropped.most_common(30) or "none")
print(f"\n{len(digits)} verses contain digits. First few:")
for (b, c), v, raw in digits[:5]:
    print(f"  {b} {c}:{v}  {raw[:100]}")
    print(f"  {'':>9}-> {clean_text(raw, alphabet, strip)[:100]}")

## 2. Find the audio

This lists every audio link on the pages in `source.audio_pages`, then works
out which book and chapter each file is from its name. **Check the matches by
eye.** If they are wrong, or many files are unmatched, write a regex for
`source.audio_filename_regex` (the unit tests in `tests/test_corpus.py` show
an example) and rerun.

If the site offers only a single zip of everything, put its URL in
`source.audio_zip_url` and use the second cell instead.

In [ ]:
from collections import Counter
from bibletts.download import find_links, catalogue

links = []
for page in cfg["source"]["audio_pages"]:
    try:
        found = find_links(page)
        print(f"{page}: {len(found)} links")
        links += found
    except Exception as e:
        print(f"{page}: failed ({e})")
links = list(dict.fromkeys(links))
print("\nBy file type:", Counter(Path(l).suffix.lower() for l in links))
print("\nZip files:", *[l for l in links if l.lower().endswith(".zip")] or ["none"], sep="\n  ")
print("\nFirst audio links:", *[l for l in links if not l.lower().endswith(".zip")][:15], sep="\n  ")

In [ ]:
FORMAT = ".mp3"   # or ".ogg": pick the one with the higher bitrate after section 3
regex = cfg["source"]["audio_filename_regex"]

if cfg["source"]["audio_zip_url"]:
    # One big zip: download it once to Drive, then work with the names inside.
    from bibletts.download import zip_listing
    audio_zip = download(cfg["source"]["audio_zip_url"], RAW / "audio_zip" / "all_audio.zip")
    names = [n for n in zip_listing(audio_zip) if n.lower().endswith(FORMAT)]
else:
    names = [l for l in links if l.lower().endswith(FORMAT)]

audio_by_chapter, unmatched = catalogue(names, regex)
print(f"Matched {len(audio_by_chapter)} chapters, {len(unmatched)} names unmatched.")
print("\nSpot-check these matches:")
for key in list(audio_by_chapter)[:: max(1, len(audio_by_chapter) // 12)][:12]:
    print(f"  {key[0]} {key[1]:>3}  <-  {Path(audio_by_chapter[key]).name}")
print("\nUnmatched:", *unmatched[:15] or ["none"], sep="\n  ")

In [ ]:
from bibletts.corpus import resolve_splits, chapter_id

splits = resolve_splits(cfg["splits"])
missing_audio = [k for k in splits if k not in audio_by_chapter]
missing_text = [k for k in splits if k not in chapters_text]
print(f"{len(splits)} chapters in the splits.")
print("Missing audio:", missing_audio or "none")
print("Missing text:", missing_text or "none")

In [ ]:
import json, shutil, zipfile

def fetch_chapter(key):
    """Download (or unzip) one chapter's audio to Drive. Returns the local path."""
    src = audio_by_chapter[key]
    dest = RAW / "audio" / f"{chapter_id(*key)}{Path(src).suffix.lower()}"
    if dest.exists():
        return dest
    if cfg["source"]["audio_zip_url"]:
        dest.parent.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(audio_zip) as z, z.open(src) as fin, open(dest, "wb") as fout:
            shutil.copyfileobj(fin, fout)
        return dest
    return download(src, dest)

# Three chapters to listen to first: narrative, sermon, poetry.
samples = {k: fetch_chapter(k) for k in [("MAT", 1), ("MAT", 5), ("PSA", 23)] if k in audio_by_chapter}
print(*samples.values(), sep="\n")

## 3. Listen and inspect

Listen to the start of each chapter and follow along in the text. Things to note:

- Is it **one reader** the whole way through?
- Is there **music**, a jingle, or a sound effect at the start or end?
- Does the reader **announce the chapter** ("Matthew, chapter one")? If so,
  write the exact wording; it goes in `text.chapter_announcement`.
- Does the reader say anything **not in the text** (section headings,
  "verse one", footnotes) or skip anything?
- **Background noise**, hiss, room echo, clipping?

In [ ]:
import pandas as pd
from IPython.display import Audio, display
from bibletts.audio import probe, load_audio

print(pd.DataFrame({chapter_id(*k): probe(p) for k, p in samples.items()}).T)

for key, path in samples.items():
    info = probe(path)
    clip = load_audio(path, info["sample_rate"], 0, 40)  # first 40 seconds
    print(f"\n{key[0]} {key[1]}: first 40 s. Text of the first verses:")
    for v, raw in chapters_text[key][:3]:
        print(f"  {v}  {raw}")
    display(Audio(clip, rate=info["sample_rate"]))

In [ ]:
# The end of each chapter: listen for music, outros, or the next announcement.
for key, path in samples.items():
    info = probe(path)
    clip = load_audio(path, info["sample_rate"], max(0, info["duration_s"] - 20))
    print(f"{key[0]} {key[1]}: last 20 s. Last verse: {chapters_text[key][-1][1]}")
    display(Audio(clip, rate=info["sample_rate"]))

## 4. Where does the frequency content stop?

MP3 encoders often remove everything above some frequency to save space. If the
recording has nothing above 8 kHz, a 16 kHz sample rate loses nothing and makes
every model cheaper. The mel-spectrogram uses up to 8 kHz (`audio.fmax`), so
content must reach at least that far for the 22,050 Hz setting to be worth it.

In [ ]:
import matplotlib.pyplot as plt
from bibletts.audio import average_spectrum_db, bandwidth_hz
from bibletts.viz import plot_spectrum

key = ("MAT", 5) if ("MAT", 5) in samples else next(iter(samples))
info = probe(samples[key])
start = min(30, info["duration_s"] / 4)                     # skip any intro
x = load_audio(samples[key], info["sample_rate"], start, 120)  # up to 2 minutes of speech
bw = bandwidth_hz(x, info["sample_rate"])
plot_spectrum(*average_spectrum_db(x, info["sample_rate"]), bandwidth=bw,
              title=f"{key[0]} {key[1]} average spectrum ({info['codec']}, {info['bit_rate_kbps']} kb/s)")
plt.show()
print(f"Content ends around {bw/1000:.1f} kHz.",
      "Keep 22,050 Hz." if bw >= 8500 else "Consider sample_rate: 16000 in configs/data.yaml.")

## 5. Download every chapter in the splits and count the hours

This saves the original files to Drive (`raw/audio/`), so milestone 1 can start
from them in any later session. Expect a few hundred MB.

In [ ]:
from tqdm.auto import tqdm

local_audio = {}
for key in tqdm([k for k in splits if k in audio_by_chapter]):
    local_audio[key] = fetch_chapter(key)

durations = {k: probe(p)["duration_s"] for k, p in tqdm(local_audio.items(), desc="measuring")}

# Record where everything is, for notebook 01.
index = {
    "vpl": str(vpl_path),
    "audio": {chapter_id(*k): str(p) for k, p in local_audio.items()},
}
(RAW / "sources.json").write_text(json.dumps(index, indent=1))
print("Saved", RAW / "sources.json")

In [ ]:
rows = [{"split": splits[k], "book": k[0], "chapter": k[1], "minutes": d / 60}
        for k, d in durations.items()]
df = pd.DataFrame(rows)
print("Hours per split:")
print((df.groupby("split")["minutes"].sum() / 60).round(2).to_string())
print("\nHours per book and split:")
print((df.groupby(["split", "book"])["minutes"].sum() / 60).round(2).to_string())
print(f"\nMean chapter length: {df.minutes.mean():.1f} min, longest: {df.minutes.max():.1f} min")

Adjust the `train` list in `configs/data.yaml` so the train split is about 10
hours. Remember that milestone 2 will drop some audio (silences at clip edges,
badly aligned clips), so aim slightly above 10.

## 6. Mel-spectrogram round trip

The TTS model will predict mel-spectrograms, not audio. This checks two things:
that the mel settings in the config keep the words intelligible, and what
Griffin-Lim (turning a mel-spectrogram back into audio with no learned model)
sounds like. Expect a buzzy, metallic voice. **Every word should still be
clear.** If it is, the mel-spectrogram holds what the TTS model needs, and the
vocoder in milestone 4 only has to make it sound natural.

In [ ]:
import torch
from bibletts.audio import MelConfig, MelSpectrogram
from bibletts.viz import plot_mel

mel_cfg = MelConfig.from_dict(cfg["audio"])
mel = MelSpectrogram(mel_cfg)

clip = load_audio(samples[key], mel_cfg.sample_rate, start, 10)   # 10 s of speech
log_mel = mel(clip)
print("Audio samples:", len(clip), "-> mel frames:", log_mel.shape[1],
      f"({log_mel.shape[0]} bands, one frame per {1000 * mel_cfg.hop_length / mel_cfg.sample_rate:.1f} ms)")
plot_mel(log_mel, mel_cfg.sample_rate, mel_cfg.hop_length, title="log mel-spectrogram")
plt.show()

print("Original:")
display(Audio(clip, rate=mel_cfg.sample_rate))
for n_iter in (5, 60):
    print(f"Griffin-Lim, {n_iter} iterations:")
    display(Audio(mel.griffin_lim(log_mel, n_iter=n_iter), rate=mel_cfg.sample_rate))

## 7. Notes (fill in before milestone 1)

Edit this cell. Then update `configs/data.yaml` and commit it.

- **Reader:** one person throughout? yes / no
- **Chapter announcement:** exact wording, or none
- **Music or other non-speech:** where, how long
- **Text differences:** anything spoken but not written, or written but not spoken
- **Audio quality:** noise, echo, clipping
- **Format chosen:** mp3 / ogg, bitrate, original sample rate
- **Content ends at:** __ kHz → sample rate __
- **Hours per split:** train __, dev __, test_same __, test_style __, test_names __
- **Config changes made:**